# Know Your Batik — Model Evaluation

In [2]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import sys
import pickle
from pathlib import Path

import torch
import yaml

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import get_dataloaders
from src.model import get_model

with open(ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

PROCESSED_PATH  = ROOT / cfg['data']['processed_path']
NUM_CLASSES     = cfg['data']['num_classes']
BATCH_SIZE      = cfg['model']['batch_size']
MODELS_DIR      = ROOT / 'models'
OUTPUTS_DIR     = ROOT / 'outputs'
CHECKPOINT_PATH = MODELS_DIR / 'checkpoint_best.pth'
LABELS_PATH     = MODELS_DIR / 'class_labels.pkl'

OUTPUTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


## 1. Load Model

In [4]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
print(checkpoint.keys())

dict_keys(['epoch', 'model_state', 'optimizer_state', 'best_val_acc'])


In [5]:
# ── Cell 2: Load class labels and model ────────────────────────────────────
with open(LABELS_PATH, 'rb') as f:
    label_data = pickle.load(f)

idx_to_class = label_data['idx_to_class']
class_names  = [idx_to_class[i] for i in range(NUM_CLASSES)]

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model = get_model(num_classes=NUM_CLASSES)
model.load_state_dict(checkpoint['model_state'])
model.eval()
model.to(DEVICE)

epoch    = checkpoint.get('epoch', 'N/A')
best_acc = checkpoint.get('best_val_acc', checkpoint.get('val_acc', 'N/A'))
if isinstance(best_acc, float):
    best_acc = f'{best_acc:.4f}'

print(f'Loaded from epoch {epoch}, best_val_acc {best_acc}')

Loaded from epoch 15, best_val_acc 0.9108


## 2. Run Inference on Test Set

In [6]:
# ── Cell 3: Inference ──────────────────────────────────────────────────────
_, _, test_loader = get_dataloaders(
    processed_path=PROCESSED_PATH,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Test accuracy: {test_acc:.4f}')

Class mapping saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\models\class_labels.pkl
Train: 1701  Val: 213  Test: 213


c:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test accuracy: 0.8592


## 3. Classification Report

In [7]:
# ── Cell 4: Classification report ─────────────────────────────────────────
from sklearn.metrics import classification_report

report = classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=4,
)
print(report)

report_path = OUTPUTS_DIR / 'classification_report.txt'
with open(report_path, 'w') as f:
    f.write(report)
print(f'Saved → {report_path}')

                             precision    recall  f1-score   support

                Bali_Barong     0.8333    1.0000    0.9091         5
                 Bali_Merak     0.6667    0.8000    0.7273         5
                     Ceplok     1.0000    0.4000    0.5714         5
               Corak_Insang     0.8889    1.0000    0.9412         8
                 Ikat_Celup     1.0000    0.7500    0.8571         4
        Jakarta_Ondel_Ondel     1.0000    0.6667    0.8000         6
     Jawa_Barat_Megamendung     1.0000    0.9500    0.9744        20
           Jawa_Timur_Pring     0.7500    1.0000    0.8571         6
           Kalimantan_Dayak     0.8500    0.8947    0.8718        19
              Lampung_Gajah     1.0000    1.0000    1.0000         6
                      Lasem     0.5000    0.6667    0.5714         6
         Madura_Mataketeran     1.0000    1.0000    1.0000         7
                Maluku_Pala     0.8750    1.0000    0.9333         7
                NTB_Lumbung     1

## 4. Confusion Matrix

In [8]:
# ── Cell 5: Confusion matrix heatmap ──────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()

cm_path = OUTPUTS_DIR / 'confusion_matrix.png'
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {cm_path}')

Saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\outputs\confusion_matrix.png


## 5. Per-Class Accuracy Bar Chart

In [9]:
# ── Cell 6: Per-class accuracy bar chart ──────────────────────────────────
import numpy as np

per_class_acc = cm.diagonal() / cm.sum(axis=1)
sorted_idx    = np.argsort(per_class_acc)          # ascending: worst → best
sorted_acc    = per_class_acc[sorted_idx]
sorted_names  = [class_names[i] for i in sorted_idx]

colors = [
    'green' if a > 0.90 else ('orange' if a >= 0.80 else 'red')
    for a in sorted_acc
]

mean_acc = per_class_acc.mean()

fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(sorted_names, sorted_acc, color=colors)
ax.axvline(mean_acc, color='red', linestyle='--', linewidth=1.5,
           label=f'Mean acc = {mean_acc:.4f}')
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Per-Class Accuracy (sorted ascending)', fontsize=13)
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()

pca_path = OUTPUTS_DIR / 'per_class_accuracy.png'
fig.savefig(pca_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {pca_path}')

Saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\outputs\per_class_accuracy.png


## 6. Top-5 Confused Pairs

In [10]:
# ── Cell 7: Top-5 confused pairs ──────────────────────────────────────────
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)          # zero out correct predictions

flat_idx   = np.argsort(cm_off, axis=None)[::-1][:5]
true_idx, pred_idx = np.unravel_index(flat_idx, cm_off.shape)

header = f"{'True Class':<30} {'Predicted As':<30} {'Count':>6}"
print(header)
print('-' * len(header))
top_confused_pair = None
for rank, (t, p) in enumerate(zip(true_idx, pred_idx)):
    count = cm_off[t, p]
    print(f"{class_names[t]:<30} {class_names[p]:<30} {count:>6}")
    if rank == 0:
        top_confused_pair = (class_names[t], class_names[p], count)

True Class                     Predicted As                    Count
--------------------------------------------------------------------
Yogyakarta_Parang              Solo_Parang                         5
Solo_Parang                    Yogyakarta_Parang                   2
Priangan_Merak_Ngibing         Lasem                               2
Papua_Cendrawasih              Lasem                               1
Sekar                          Papua_Cendrawasih                   1


## 7. Summary

In [11]:
# ── Cell 8: Summary ────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, classification_report as cr

macro_f1    = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

# Per-class F1 for best/worst
per_class_f1 = f1_score(all_labels, all_preds, average=None)
best_cls_idx  = int(np.argmax(per_class_f1))
worst_cls_idx = int(np.argmin(per_class_f1))

print('=' * 50)
print('EVALUATION SUMMARY')
print('=' * 50)
print(f'Test Accuracy       : {test_acc:.4f}')
print(f'Macro F1            : {macro_f1:.4f}')
print(f'Weighted F1         : {weighted_f1:.4f}')
print(f'Best  class (F1)    : {class_names[best_cls_idx]} ({per_class_f1[best_cls_idx]:.4f})')
print(f'Worst class (F1)    : {class_names[worst_cls_idx]} ({per_class_f1[worst_cls_idx]:.4f})')
if top_confused_pair:
    print(f'Top confused pair   : "{top_confused_pair[0]}" → "{top_confused_pair[1]}" ({top_confused_pair[2]} times)')
print('=' * 50)

EVALUATION SUMMARY
Test Accuracy       : 0.8592
Macro F1            : 0.8321
Weighted F1         : 0.8559
Best  class (F1)    : Lampung_Gajah (1.0000)
Worst class (F1)    : Priangan_Merak_Ngibing (0.4000)
Top confused pair   : "Yogyakarta_Parang" → "Solo_Parang" (5 times)
